# ECM frame sharing on Wan 2.1 T2V 1.3B
Re-noised averaged frame sharing vs fewer sampling steps, measured against the full-compute output with identical noise. Needs: GPU T4 x2, Internet on. Outputs land in /kaggle/working/ecm_wan.

In [ ]:
!pip -q install diffusers==0.42.1 transformers accelerate ftfy lpips imageio-ffmpeg scikit-image 2>&1 | tail -2
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile ecm_wan.py
"""Re-noised averaged frame sharing for Wan 2.1 (diffusers WanTransformer3DModel).

The method (same as the toy study, experiments/r3f_battery.py::avg_sample):
  1. Run the first `warm` sampling steps at full compute. From the model's clean-latent estimate
     x0_hat at that point, mark spatial token positions whose content barely changes across latent
     frames ("static"). Optionally disable sharing if the whole frame is translating (camera pan).
  2. For the remaining steps, every static position forms a group: its tokens in latent frames
     `first..F-1`. Each token keeps its OWN noisy state. The group's leader gets the average of the
     group's states plus sigma*sqrt(1-1/m) fresh noise (restoring the per-token noise level), and a
     log(m) key bias in self-attention (so the leader carries the attention weight of m copies).
     Only leaders and non-static tokens go through the transformer.
  3. The leader's clean estimate x0_hat is given to every token of the group; each token then steps
     with its own noise estimate eps_i = (x_i - (1 - sigma) x0_hat) / sigma.

Conventions (diffusers Wan / flow matching): x_sigma = (1 - sigma) x0 + sigma eps, model output
v = eps - x0, so x0_hat = x - sigma v.  Euler step: x <- x + (sigma_next - sigma) v.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F


# ----------------------------------------------------------------------------- token layout

def patchify(lat, p):
    """(B,C,F,H,W) -> (B, N, pt*ph*pw*C), token order and channel layout identical to proj_out's."""
    B, C, Fr, H, W = lat.shape
    pt, ph, pw = p
    x = lat.reshape(B, C, Fr // pt, pt, H // ph, ph, W // pw, pw)
    x = x.permute(0, 2, 4, 6, 3, 5, 7, 1)          # B, f, h, w, pt, ph, pw, C
    return x.reshape(B, (Fr // pt) * (H // ph) * (W // pw), pt * ph * pw * C)


def unpatchify(tok, p, shape):
    B, C, Fr, H, W = shape
    pt, ph, pw = p
    x = tok.reshape(B, Fr // pt, H // ph, W // pw, pt, ph, pw, C)
    x = x.permute(0, 7, 1, 4, 2, 5, 3, 6)
    return x.flatten(6, 7).flatten(4, 5).flatten(2, 3)


# ----------------------------------------------------------------------------- kept-token forward

def _rope(x, cos, sin):
    x1, x2 = x.unflatten(-1, (-1, 2)).unbind(-1)
    c, s = cos[..., 0::2], sin[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * c - x2 * s
    out[..., 1::2] = x1 * s + x2 * c
    return out.type_as(x)


def self_attn(attn, h, rope, key_bias):
    """WanAttnProcessor self-attention with an additive per-key bias (B or 1, K)."""
    if attn.fused_projections:
        q, k, v = attn.to_qkv(h).chunk(3, dim=-1)
    else:
        q, k, v = attn.to_q(h), attn.to_k(h), attn.to_v(h)
    q, k = attn.norm_q(q), attn.norm_k(k)
    q, k, v = (t.unflatten(2, (attn.heads, -1)) for t in (q, k, v))
    q, k = _rope(q, *rope), _rope(k, *rope)
    D = q.shape[-1]
    if key_bias is not None:
        # Exact, mask-free additive key bias: append 8 channels; q gets 1 in the first, k gets
        # bias*sqrt(D), so with scale 1/sqrt(D) the logit gains exactly `bias`. A float attn_mask
        # would push SDPA onto kernels that materialise the full L x L matrix (~9 GB here).
        B, L, H, _ = q.shape
        qe = q.new_zeros(B, L, H, 8); qe[..., 0] = 1
        ke = k.new_zeros(k.shape[0], k.shape[1], H, 8)
        ke[..., 0] = (key_bias.to(torch.float32) * math.sqrt(D)).to(k.dtype)[:, :, None]
        q, k, v = torch.cat([q, qe], -1), torch.cat([k, ke], -1), torch.cat([v, torch.zeros_like(ke)], -1)
    o = F.scaled_dot_product_attention(q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), scale=1 / math.sqrt(D))
    o = o[..., :D].transpose(1, 2).flatten(2, 3).type_as(h)
    o = attn.to_out[0](o)
    return attn.to_out[1](o)


def block_forward(blk, h, enc, temb, rope, key_bias):
    """h is kept in fp32 (diffusers keeps it in the model dtype); module inputs are cast to the model
    dtype. Guards against fp16 overflow on GPUs without bf16 (Wan was trained in bf16)."""
    dt = enc.dtype
    sh, sc, g, csh, csc, cg = (blk.scale_shift_table + temb.float()).chunk(6, dim=1)
    h = h + self_attn(blk.attn1, (blk.norm1(h) * (1 + sc) + sh).to(dt), rope, key_bias).float() * g
    h = h + blk.attn2(blk.norm2(h).to(dt), enc, None, None).float()
    return h + blk.ffn((blk.norm3(h) * (1 + csc) + csh).to(dt)).float() * cg


@torch.no_grad()
def forward_tokens(model, lat_shape, tok, keep, timestep, enc, key_bias=None):
    """Run WanTransformer3DModel on a subset of tokens.

    tok: (B, K, pt*ph*pw*C) latent tokens at positions `keep` (LongTensor of K indices into the full
    token grid). Returns the model output for those tokens, same layout. With keep = all positions and
    key_bias = None this equals model(...) exactly (checked in test_ecm_wan.py)."""
    p = model.config.patch_size
    B = tok.shape[0]
    full_cos, full_sin = model.rope(torch.empty(lat_shape, device=tok.device))
    rope = (full_cos[:, keep], full_sin[:, keep])
    # patch embedding is a strided conv == a linear map on each token
    W = model.patch_embedding.weight                                    # D, C, pt, ph, pw
    Wl = W.permute(0, 2, 3, 4, 1).reshape(W.shape[0], -1)               # D, pt*ph*pw*C (token layout)
    h = F.linear(tok.to(W.dtype), Wl, model.patch_embedding.bias).float()
    temb, tproj, enc, _ = model.condition_embedder(timestep, enc, None, timestep_seq_len=None)
    tproj = tproj.unflatten(1, (6, -1))
    for blk in model.blocks:
        h = block_forward(blk, h, enc, tproj, rope, key_bias)
    sh, sc = (model.scale_shift_table.to(temb.device) + temb.unsqueeze(1)).chunk(2, dim=1)
    h = (model.norm_out(h) * (1 + sc) + sh).to(model.proj_out.weight.dtype)
    return model.proj_out(h)


# ----------------------------------------------------------------------------- schedule & sampling

def flow_sigmas(n_steps, shift):
    s = torch.linspace(1.0, 1.0 / 1000, n_steps, dtype=torch.float64)
    s = shift * s / (1 + (shift - 1) * s)
    return torch.cat([s, torch.zeros(1, dtype=torch.float64)])


def _guided(model, shape, tok, keep, sigma, enc_c, enc_u, gscale, key_bias):
    """CFG velocity for kept tokens; cond & uncond batched together."""
    B = tok.shape[0]
    ts = torch.full((2 * B,), float(sigma) * 1000, device=tok.device)
    kb = None if key_bias is None else key_bias.expand(2 * B, -1)
    v = forward_tokens(model, shape, torch.cat([tok, tok]), keep, ts, torch.cat([enc_c, enc_u]), kb).float()
    assert torch.isfinite(v).all(), "non-finite model output (fp16 overflow?)"
    vc, vu = v[:B], v[B:]
    return vu + gscale * (vc - vu)


def static_map(x0_tok, grid, thr, first_frame, pan_gate=None, frac=None):
    """x0_tok: (1, N, D) clean-latent estimate. grid = (Fl, Hl, Wl) token grid.
    A position is static if the max over consecutive latent frames (from first_frame on) of the RMS
    token change, divided by the RMS token magnitude, is below thr. Returns (src, info)."""
    Fl, Hl, Wl = grid
    x = x0_tok[0].detach().float().cpu().reshape(Fl, Hl * Wl, -1)[first_frame:]   # map is built on CPU
    scale = x.pow(2).mean().sqrt() + 1e-8
    act = (x[1:] - x[:-1]).pow(2).mean(-1).sqrt().amax(0) / scale       # Hl*Wl
    stat = (act <= torch.quantile(act.float(), frac)) if frac is not None else (act < thr)
    pan = 0.0
    if pan_gate is not None:
        pan = global_shift(x[0].mean(-1).reshape(Hl, Wl), x[-1].mean(-1).reshape(Hl, Wl)) / max(1, x.shape[0] - 1)
        if pan > pan_gate:
            stat[:] = False
    src = torch.arange(Fl * Hl * Wl)
    pos = torch.nonzero(stat).flatten()
    leader = first_frame * Hl * Wl + pos
    for f in range(first_frame + 1, Fl):
        src[f * Hl * Wl + pos] = leader
    return src, dict(static_frac=float(stat.float().mean()), pan=float(pan), act=act)


def global_shift(a, b):
    """Phase-correlation translation (in tokens) between two 2-D maps."""
    A, Bf = torch.fft.fft2(a - a.mean()), torch.fft.fft2(b - b.mean())
    R = A * Bf.conj()
    r = torch.fft.ifft2(R / (R.abs() + 1e-8)).real
    i = int(torch.argmax(r))
    iy, ix = divmod(i, r.shape[1])

    def sub(c, l, rr):            # parabolic peak refinement
        den = l - 2 * c + rr
        return 0.0 if abs(float(den)) < 1e-12 else float(0.5 * (l - rr) / den)
    H, W = r.shape
    dy = iy + sub(r[iy, ix], r[(iy - 1) % H, ix], r[(iy + 1) % H, ix])
    dx = ix + sub(r[iy, ix], r[iy, (ix - 1) % W], r[iy, (ix + 1) % W])
    dy = dy - H if dy > H / 2 else dy
    dx = dx - W if dx > W / 2 else dx
    return math.hypot(dy, dx)


@torch.no_grad()
def sample(model, noise, enc_c, enc_u, n_steps=30, shift=3.0, gscale=5.0, share=None, seed=0, callback=None):
    """Euler flow sampler. share=None -> plain sampling. share=dict(thr, warm, first_frame, pan_gate,
    renoise=True, average=True, prop=True) -> re-noised averaged frame sharing.
    Returns (latents, stats) where stats counts token-forwards (compute) and sharing info."""
    p = model.config.patch_size
    shape = noise.shape
    grid = (shape[2] // p[0], shape[3] // p[1], shape[4] // p[2])
    N = grid[0] * grid[1] * grid[2]
    x = patchify(noise.float(), p)
    sig = flow_sigmas(n_steps, shift)
    allk = torch.arange(N, device=x.device)
    work, info = 0, {}
    gen = torch.Generator(device=x.device).manual_seed(seed)
    keep = inv = mult = kb = None
    warm = None

    def build(x0_est):
        nonlocal keep, inv, mult, kb, info
        src, info = static_map(x0_est, grid, share.get("thr", 0.0), share.get("first_frame", 1), share.get("pan_gate"),
                               share.get("frac"))
        kp = torch.unique(src).to(x.device)
        info["kept_frac"] = len(kp) / N
        if len(kp) < N:            # otherwise nothing is shared: stay exactly on plain sampling
            keep = kp
            inv = torch.searchsorted(keep, src.to(x.device))
            mult = torch.bincount(inv, minlength=len(keep)).float()
            kb = torch.log(mult)[None] if share.get("prop", True) else None

    if share is not None and share.get("draft"):
        # toy-study protocol: a cheap separate full-compute draft from the SAME noise gives a complete
        # (blurry) video; its motion decides the map, and sharing then runs from step 0.
        d_lat, d_info = sample(model, noise, enc_c, enc_u, n_steps=share["draft"], shift=shift, gscale=gscale)
        work += d_info["token_forwards"]
        build(patchify(d_lat.float(), p))
        info["draft_steps"] = share["draft"]
        warm = 0
    elif share is not None:   # number of full-compute steps before the map is built
        warm = share.get("warm") or max(1, int((sig[:-1] > share.get("warm_sigma", 0.85)).sum()))
    for k in range(n_steps):
        s, s2 = float(sig[k]), float(sig[k + 1])
        if share is None or k < warm or keep is None:
            v = _guided(model, shape, x.to(model.dtype), allk, s, enc_c, enc_u, gscale, None)
            work += N
            x0 = x - s * v
            x = x + (s2 - s) * v
            if share is not None and k == warm - 1:
                build(x0)
                info["warm_steps"] = warm
        else:
            if share.get("average", True):
                z = torch.zeros(x.shape[0], len(keep), x.shape[-1], device=x.device).index_add_(1, inv, x) / mult[None, :, None]
                if share.get("renoise", True):
                    z = z + s * torch.sqrt(1 - 1 / mult)[None, :, None] * torch.randn(z.shape, generator=gen, device=x.device)
            else:
                z = x[:, keep]
            v = _guided(model, shape, z.to(model.dtype), keep, s, enc_c, enc_u, gscale, kb)
            work += len(keep)
            x0 = (z - s * v)[:, inv]                       # leader's clean estimate for every member
            eps = (x - (1 - s) * x0) / s                    # each token keeps its own noise estimate
            x = (1 - s2) * x0 + s2 * eps
        if callback is not None:
            callback(k, x, x0)
    info["token_forwards"] = work
    info["full_token_forwards"] = N * n_steps
    return unpatchify(x, p, shape), info


@torch.no_grad()
def sample_unipc(model, noise, enc_c, enc_u, n_steps=15, shift=3.0, gscale=5.0):
    """Fewer-steps baseline with the scheduler Wan's own pipeline uses (UniPC, flow sigmas)."""
    from diffusers import UniPCMultistepScheduler
    sch = UniPCMultistepScheduler(prediction_type="flow_prediction", use_flow_sigmas=True, flow_shift=shift,
                                  num_train_timesteps=1000)
    sch.set_timesteps(n_steps, device=noise.device)
    p = model.config.patch_size
    shape = noise.shape
    N = (shape[2] // p[0]) * (shape[3] // p[1]) * (shape[4] // p[2])
    allk = torch.arange(N, device=noise.device)
    x = noise.float()
    for t in sch.timesteps:
        v = _guided(model, shape, patchify(x, p).to(model.dtype), allk, float(t) / 1000, enc_c, enc_u, gscale, None)
        x = sch.step(unpatchify(v, p, shape), t, x, return_dict=False)[0]
    return x, dict(token_forwards=N * n_steps, full_token_forwards=N * n_steps)


In [ ]:
%%writefile kaggle_run.py
"""Decisive real-model experiment: re-noised averaged frame sharing on Wan 2.1 T2V 1.3B.

Designed for a free Kaggle notebook (2x T4 16GB). Steps:
  python kaggle_run.py encode                 # T5 prompt embeddings on CPU (bf16) -> embeds.pt
  python kaggle_run.py gen --gpu 0 --shard 0/2 &  python kaggle_run.py gen --gpu 1 --shard 1/2
  python kaggle_run.py report                 # metrics vs the full-compute reference -> results.md

Every method for a prompt uses the same initial noise, so outputs are directly comparable with the
full-compute reference (the standard protocol of training-free acceleration papers: PSNR/SSIM/LPIPS
to the original model's output, plus wall-clock). Resumable: finished videos are skipped.
Use --dry to run the whole pipeline on CPU with a tiny random-weight transformer (no downloads).
"""
import argparse, json, os, sys, time

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import ecm_wan as E

MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
NEG = ("Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall "
       "gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, "
       "poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three "
       "legs, many people in the background, walking backwards")

# 16 static/locked-off camera scenes (where sharing should help) + 8 moving-camera scenes (gate test)
PROMPTS_STATIC = [
    "A cat sitting on a windowsill, its tail slowly swishing, soft afternoon light, locked-off camera",
    "A candle flame flickering on a wooden table in a dark room, static shot",
    "A barista pouring milk into a cup of coffee, close-up, tripod shot",
    "A goldfish swimming in a round glass bowl on a kitchen counter, static camera",
    "A man reading a newspaper on a park bench, pigeons walking nearby, fixed camera",
    "Rain drops falling into a puddle on a street at night, neon reflections, static shot",
    "A woman typing on a laptop in a quiet library, medium shot, static camera",
    "A red ball bouncing on a tiled floor in an empty room, static shot",
    "A dog wagging its tail on a front porch, fixed camera",
    "Steam rising from a bowl of ramen on a restaurant table, static close-up",
    "A clock with a moving second hand on a white wall, static shot",
    "Leaves of a potted plant moving gently in the breeze near an open window, static camera",
    "A chef chopping vegetables on a cutting board, overhead static shot",
    "A child blowing soap bubbles in a backyard, fixed camera",
    "Waves gently lapping at a rocky shore, static wide shot",
    "A person walking across a crosswalk in front of a stationary camera, city street",
]
PROMPTS_MOVING = [
    "Camera slowly pans right across a mountain landscape at sunrise",
    "Drone shot flying forward over a forest river",
    "Tracking shot following a cyclist on a coastal road",
    "Camera dolly moving through a busy market street",
    "Slow pan across a bookshelf full of old books",
    "Handheld camera walking through a subway station",
    "Aerial shot circling a lighthouse on a cliff",
    "Camera tilting up a tall glass skyscraper",
]
# interleave 2 static : 1 moving so a run cut short by the time budget still covers both kinds
PROMPTS = []
for j in range(8):
    PROMPTS += [("static", PROMPTS_STATIC[2 * j]), ("static", PROMPTS_STATIC[2 * j + 1]), ("moving", PROMPTS_MOVING[j])]


def methods(steps):
    """name -> (n_steps, share-config | None | "unipc").
    Sharing picks the `frac` least-active token positions (quantile, so the sweep spans the range).
    The map comes either from a cheap separate draft from the same noise (draft=3: the toy-study
    protocol, sharing from step 0) or from the model's own estimate after the steps with sigma > 0.85
    (warm: no extra pass, but sharing starts late, which caps the speedup)."""
    M = {f"full_{steps}": (steps, None)}
    for n in [steps * 2 // 3, steps // 2, steps // 3]:          # 20 / 15 / 10 at steps=30
        M[f"steps_{n}"] = (n, None)
    for n in [steps * 2 // 3, steps // 2]:
        M[f"unipc_{n}"] = (n, "unipc")
    d = dict(first_frame=1, pan_gate=0.25, draft=3)
    for frac in [0.5, 0.75, 0.9]:
        M[f"share_{steps}_f{frac}_draft3"] = (steps, dict(d, frac=frac))
    M[f"share_{steps}_f0.75_warm"] = (steps, dict(first_frame=1, pan_gate=0.25, warm_sigma=0.85, frac=0.75))
    M[f"share_{steps // 2}_f0.75_draft3"] = (steps // 2, dict(d, frac=0.75))         # stacking on fewer steps
    M[f"abl_noRenoise_{steps}_f0.75_draft3"] = (steps, dict(d, frac=0.75, renoise=False))
    M[f"abl_noBias_{steps}_f0.75_draft3"] = (steps, dict(d, frac=0.75, prop=False))
    M[f"abl_noGate_{steps}_f0.75_draft3"] = (steps, dict(d, frac=0.75, pan_gate=None))   # also a guard if the gate misfires
    return M


# ----------------------------------------------------------------------------- model loading

def load_transformer(args, device, dtype):
    if args.dry:
        from diffusers import WanTransformer3DModel
        torch.manual_seed(0)
        m = WanTransformer3DModel(patch_size=(1, 2, 2), num_attention_heads=2, attention_head_dim=24, in_channels=16,
                                  out_channels=16, text_dim=64, freq_dim=32, ffn_dim=96, num_layers=2,
                                  cross_attn_norm=True, rope_max_seq_len=1024)
        with torch.no_grad():
            for p in m.parameters():
                p.add_(torch.randn_like(p) * 0.02)
        return m.to(device, dtype).eval()
    from diffusers import WanTransformer3DModel
    return WanTransformer3DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype).to(device).eval()


def load_vae(args, device):
    if args.dry:
        return None
    from diffusers import AutoencoderKLWan
    return AutoencoderKLWan.from_pretrained(MODEL, subfolder="vae", torch_dtype=torch.float32).to(device).eval()


@torch.no_grad()
def decode(vae, lat, args):
    """latents -> uint8 video (F,H,W,3)."""
    if vae is None:   # dry run: fake 'decode' by nearest upsampling of 3 latent channels
        v = lat[0, :3].float().clamp(-2, 2) / 4 + 0.5
        v = v.repeat_interleave(4, 1)[:, :args.frames].repeat_interleave(8, 2).repeat_interleave(8, 3)
        return (v.permute(1, 2, 3, 0).cpu().numpy() * 255).astype(np.uint8)
    mean = torch.tensor(vae.config.latents_mean).view(1, -1, 1, 1, 1).to(lat)
    std = torch.tensor(vae.config.latents_std).view(1, -1, 1, 1, 1).to(lat)
    try:
        v = vae.decode(lat.float() * std + mean, return_dict=False)[0]      # B,3,F,H,W in [-1,1]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); vae.enable_tiling()
        v = vae.decode(lat.float() * std + mean, return_dict=False)[0]
    v = ((v[0].clamp(-1, 1) + 1) * 127.5).round().byte().permute(1, 2, 3, 0).cpu().numpy()
    return v


# ----------------------------------------------------------------------------- stages

def stage_encode(args):
    out = os.path.join(args.out, "embeds.pt")
    if os.path.exists(out):
        print("embeds exist"); return
    texts = [p for _, p in PROMPTS] + [NEG]
    if args.dry:
        g = torch.Generator().manual_seed(1)
        embs = [torch.randn(1, 12, 64, generator=g) for _ in texts]
    else:
        from transformers import AutoTokenizer, UMT5EncoderModel
        tok = AutoTokenizer.from_pretrained(MODEL, subfolder="tokenizer")
        dev = "cuda" if torch.cuda.is_available() and args.encode_on_gpu else "cpu"
        enc = UMT5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder", torch_dtype=torch.bfloat16).to(dev).eval()
        embs = []
        for i, t in enumerate(texts):
            ids = tok([t], max_length=512, truncation=True, add_special_tokens=True, return_tensors="pt")
            n = int(ids.attention_mask.sum())      # encode only the real tokens: identical to masked padding, 10x cheaper
            with torch.no_grad():
                e = enc(ids.input_ids[:, :n].to(dev), ids.attention_mask[:, :n].to(dev)).last_hidden_state[0].float().cpu()
            # pipeline convention: zero-pad to 512 tokens
            embs.append(torch.cat([e, e.new_zeros(512 - n, e.shape[1])])[None])
            print(f"encoded {i + 1}/{len(texts)}", flush=True)
    torch.save(embs, out)


def stage_gen(args):
    device = f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu"
    dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[args.dtype]
    model = load_transformer(args, device, dtype)
    vae = load_vae(args, device)
    embs = torch.load(os.path.join(args.out, "embeds.pt"))
    neg = embs[-1].to(device, dtype)
    k, n = map(int, args.shard.split("/"))
    M = methods(args.steps)
    if args.only:
        M = {a: b for a, b in M.items() if a in args.only.split(",")}
    lat_shape = (1, 16, (args.frames - 1) // 4 + 1, args.height // 8, args.width // 8)
    os.makedirs(os.path.join(args.out, "vids"), exist_ok=True)
    meta_f = os.path.join(args.out, f"meta_{k}.json")
    meta = json.load(open(meta_f)) if os.path.exists(meta_f) else {}
    idx = [i for i in range(len(PROMPTS)) if i % n == k][: args.max_prompts]
    # warm-up (CUDA kernels / allocator) so the first timed method is not penalised
    E.sample(model, torch.randn(lat_shape).to(device), embs[0].to(device, dtype), neg, n_steps=1, shift=args.shift, gscale=args.cfg)
    t_start = time.time()
    for i in idx:
        if (time.time() - t_start) / 3600 > args.hours:
            print(f"[gpu{args.gpu}] time budget reached, stopping before prompt {i}", flush=True)
            break
        cat, prompt = PROMPTS[i]
        enc = embs[i].to(device, dtype)
        noise = torch.randn(lat_shape, generator=torch.Generator().manual_seed(1000 + i)).to(device)
        for name, (steps, share) in M.items():
            key = f"{i:02d}_{name}"
            fn = os.path.join(args.out, "vids", key + ".npy")
            if key in meta and os.path.exists(fn):
                continue
            if device.startswith("cuda"):
                torch.cuda.synchronize(device)
            t0 = time.time()
            if share == "unipc":
                lat, info = E.sample_unipc(model, noise, enc, neg, n_steps=steps, shift=args.shift, gscale=args.cfg)
            else:
                lat, info = E.sample(model, noise, enc, neg, n_steps=steps, shift=args.shift, gscale=args.cfg,
                                     share=share, seed=50_000 + i)
            if device.startswith("cuda"):
                torch.cuda.synchronize(device)
            dt = time.time() - t0
            torch.save(lat.half().cpu(), fn[:-4] + ".lat.pt")      # keep the sampling work even if decode fails
            vid = decode(vae, lat, args)
            np.save(fn, vid)
            if "act" in info:   # activity distribution, for threshold calibration
                info["act_q"] = [float(q) for q in torch.quantile(info["act"].float().cpu(), torch.tensor([.1, .25, .5, .75, .9]))]
            info = {a: b for a, b in info.items() if a != "act"}
            meta[key] = dict(prompt_idx=i, category=cat, method=name, seconds=dt, **info)
            json.dump(meta, open(meta_f, "w"), indent=1)
            print(f"[gpu{args.gpu}] {key:32s} {dt:7.1f}s  compute x{info['full_token_forwards'] / info['token_forwards'] * args.steps / steps:.2f}"
                  f"  static {info.get('static_frac', 0):.2f} pan {info.get('pan', 0):.2f}", flush=True)


# ----------------------------------------------------------------------------- metrics

def psnr(a, b):
    mse = np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2)
    return 99.0 if mse == 0 else float(10 * np.log10(255.0 ** 2 / mse))


def ssim_video(a, b):
    from skimage.metrics import structural_similarity
    return float(np.mean([structural_similarity(x, y, channel_axis=2, data_range=255) for x, y in zip(a, b)]))


def flicker(v, ref):
    """Mean |frame diff| over the 10% of pixels that change least IN THE REFERENCE (fixed mask, so a
    method cannot lower its score by freezing a different set of pixels)."""
    dr = np.abs(np.diff(ref.astype(np.float32), axis=0)).mean(-1).mean(0)
    mask = dr <= np.quantile(dr, 0.1)
    d = np.abs(np.diff(v.astype(np.float32), axis=0)).mean(-1)
    return float(d[:, mask].mean())


def motion(v):
    return float(np.abs(np.diff(v.astype(np.float32), axis=0)).mean())


def stage_report(args):
    meta = {}
    for f in os.listdir(args.out):
        if f.startswith("meta_"):
            meta.update(json.load(open(os.path.join(args.out, f))))
    lp = None
    try:
        import lpips
        lp = lpips.LPIPS(net="alex", verbose=False)
    except Exception as e:  # noqa
        print("LPIPS unavailable:", e)
    clip = None
    try:
        from transformers import CLIPModel, CLIPProcessor
        clip = (CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval(),
                CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32"))
    except Exception as e:  # noqa
        print("CLIP unavailable:", e)

    def clip_score(v, prompt):
        with torch.no_grad():
            fr = [v[j] for j in np.linspace(0, len(v) - 1, 8).astype(int)]
            inp = clip[1](text=[prompt], images=fr, return_tensors="pt", padding=True)
            out = clip[0](**inp)
            im = torch.nn.functional.normalize(out.image_embeds, dim=-1)
            tx = torch.nn.functional.normalize(out.text_embeds, dim=-1)
            return float((im @ tx.T).mean() * 100)
    ref_name = f"full_{args.steps}"
    rows = {}
    for key, m in sorted(meta.items()):
        i, name = m["prompt_idx"], m["method"]
        ref_key = f"{i:02d}_{ref_name}"
        if ref_key not in meta:
            continue
        v = np.load(os.path.join(args.out, "vids", key + ".npy"))
        r = np.load(os.path.join(args.out, "vids", ref_key + ".npy"))
        rec = dict(category=m["category"], seconds=m["seconds"], ref_seconds=meta[ref_key]["seconds"],
                   work=m["token_forwards"], ref_work=meta[ref_key]["token_forwards"],
                   psnr=psnr(v, r), ssim=ssim_video(v, r), flicker=flicker(v, r), ref_flicker=flicker(r, r), motion=motion(v),
                   ref_motion=motion(r), static_frac=m.get("static_frac", 0.0), gated=m.get("pan", 0) > 0.25)
        if clip is not None:
            rec["clip"] = clip_score(v, PROMPTS[i][1])
        if lp is not None:
            with torch.no_grad():
                tv = torch.from_numpy(v).permute(0, 3, 1, 2).float() / 127.5 - 1
                tr = torch.from_numpy(r).permute(0, 3, 1, 2).float() / 127.5 - 1
                rec["lpips"] = float(np.mean([lp(tv[j:j + 8], tr[j:j + 8]).mean().item() for j in range(0, len(tv), 8)]))
        rows.setdefault(name, []).append(rec)
    json.dump(rows, open(os.path.join(args.out, "per_video.json"), "w"), indent=1)

    def agg(recs, cat):
        rs = [r for r in recs if cat == "all" or r["category"] == cat]
        if not rs:
            return None
        g = lambda k: float(np.mean([r[k] for r in rs])) if k in rs[0] else float("nan")
        return dict(n=len(rs), wall=float(np.sum([r["ref_seconds"] for r in rs]) / np.sum([r["seconds"] for r in rs])),
                    flops=float(np.sum([r["ref_work"] for r in rs]) / np.sum([r["work"] for r in rs])),
                    psnr=g("psnr"), ssim=g("ssim"), lpips=g("lpips"), flicker=g("flicker"), clip=g("clip"),
                    motion_ratio=float(np.mean([r["motion"] / max(r["ref_motion"], 1e-6) for r in rs])),
                    static=g("static_frac"), gated=g("gated"))
    lines = ["# Wan 2.1 1.3B: re-noised averaged frame sharing vs fewer steps", "",
             f"{args.frames} frames, {args.height}x{args.width}, Euler flow sampler, shift {args.shift}, CFG {args.cfg}. "
             f"Reference = {ref_name} with identical noise. Speedups: wall-clock (measured, sampling only) and "
             "token-forward ratio (compute; conservative: attention cost scales with tokens^2). Baselines steps_* use Euler, unipc_* use Wan's own UniPC scheduler. "
             "flicker uses the reference's 10% least-changing pixels. 'gated' = pan gate fired (pan > 0.25 tokens/frame). motion = mean |frame diff| relative to the reference (1.0 = same amount of motion).", ""]
    summary = {}
    for cat in ["all", "static", "moving"]:
        lines += [f"## {cat} prompts", "",
                  "| method | n | wall-clock x | compute x | PSNR | SSIM | LPIPS | CLIP | flicker | motion | static frac | gated |",
                  "|---|---|---|---|---|---|---|---|---|---|---|---|"]
        for name, recs in rows.items():
            a = agg(recs, cat)
            if a is None:
                continue
            summary.setdefault(cat, {})[name] = a
            lines.append(f"| {name} | {a['n']} | {a['wall']:.2f} | {a['flops']:.2f} | {a['psnr']:.2f} | {a['ssim']:.3f} | "
                         f"{a['lpips']:.3f} | {a['clip']:.2f} | {a['flicker']:.2f} | {a['motion_ratio']:.2f} | {a['static']:.2f} | {a['gated']:.2f} |")
        lines.append("")
    open(os.path.join(args.out, "results.md"), "w").write("\n".join(lines))
    json.dump(summary, open(os.path.join(args.out, "summary.json"), "w"), indent=1)
    print("\n".join(lines))


def stage_mp4(args):
    import imageio
    os.makedirs(os.path.join(args.out, "mp4"), exist_ok=True)
    for f in sorted(os.listdir(os.path.join(args.out, "vids"))):
        v = np.load(os.path.join(args.out, "vids", f))
        imageio.mimwrite(os.path.join(args.out, "mp4", f[:-4] + ".mp4"), list(v), fps=16, quality=8)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=["encode", "gen", "report", "mp4"])
    ap.add_argument("--out", default="/kaggle/working/ecm_wan")
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--shard", default="0/1")
    ap.add_argument("--steps", type=int, default=30)
    ap.add_argument("--frames", type=int, default=33)
    ap.add_argument("--height", type=int, default=480)
    ap.add_argument("--width", type=int, default=832)
    ap.add_argument("--shift", type=float, default=3.0)
    ap.add_argument("--cfg", type=float, default=5.0)
    ap.add_argument("--dtype", default="fp16")
    ap.add_argument("--max_prompts", type=int, default=999)
    ap.add_argument("--only", default="")
    ap.add_argument("--hours", type=float, default=8.0, help="stop starting new prompts after this many hours")
    ap.add_argument("--encode_on_gpu", action="store_true")
    ap.add_argument("--dry", action="store_true")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    {"encode": stage_encode, "gen": stage_gen, "report": stage_report, "mp4": stage_mp4}[a.stage](a)


In [ ]:
# Settings. CALIBRATE=True runs one prompt with a few methods (timing + threshold check).
CALIBRATE = False
HOURS = 7.5   # per-GPU budget for starting new prompts (Kaggle limit is 12h per run)
import torch; NGPU = max(1, torch.cuda.device_count()); print('GPUs:', NGPU)

In [ ]:
!python kaggle_run.py encode 2>&1 | tail -3

In [ ]:
import subprocess
extra = ['--max_prompts', '1', '--only', 'full_30,unipc_15,share_30_f0.75_draft3'] if CALIBRATE else []
procs = [subprocess.Popen(['python', 'kaggle_run.py', 'gen', '--gpu', str(g), '--shard', f'{g}/{NGPU}', '--hours', str(HOURS)] + extra,
                          stdout=open(f'/kaggle/working/gen_{g}.log', 'w'), stderr=subprocess.STDOUT) for g in range(NGPU)]
for p in procs: p.wait()
for g in range(NGPU): print(open(f'/kaggle/working/gen_{g}.log').read()[-6000:])

In [ ]:
!python kaggle_run.py report 2>&1 | grep -v Warning
import json, glob
for f in glob.glob('/kaggle/working/ecm_wan/meta_*.json'):
    for k, v in json.load(open(f)).items():
        print(k, round(v['seconds'], 1), 's', {a: (round(b, 3) if isinstance(b, float) else b) for a, b in v.items() if a in ('static_frac', 'pan', 'kept_frac', 'act_q')})

In [ ]:
!python kaggle_run.py mp4 && cd /kaggle/working && zip -qr ecm_wan_results.zip ecm_wan/mp4 ecm_wan/*.json ecm_wan/results.md gen_*.log && ls -la ecm_wan_results.zip
!rm -rf /kaggle/working/ecm_wan/vids   # raw frames are large; mp4s + metrics are kept